# Standalone Foundation Result Verification

This notebook independently verifies the recorded foundation result. It does not import the project package, read project source code, require a GPU, or retrain models.

It needs only this notebook, the published raw OOF CSV named foundation_ctr_tabm_base_blend_w50_oof.csv, and Python with numpy, pandas, and scikit-learn installed.

## Steps

1. Put the raw OOF CSV beside this notebook, or set OOF_PATH below.
2. Run the configuration and verification cells.
3. Read the PASS or FAIL result. The notebook writes a comparison CSV and JSON verdict in foundation-reproduction-proof.

The historical model training used an NVIDIA GeForce RTX 3060 with 12 GB VRAM. PyTorch reported 11.63 GiB total and 11.52 GiB free before training, with an 8.59 GiB preflight peak allocation. Those details are provenance only; this result-verification notebook does not need a GPU.

In [ ]:
import hashlib
import json
from pathlib import Path
import platform

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
import sklearn


In [ ]:
CANDIDATE_NAME = "foundation_ctr_tabm_base_blend_w50"
OOF_FILENAME = f"{CANDIDATE_NAME}_oof.csv"
OOF_PATH = None
OUTPUT_DIRECTORY = Path.cwd() / "foundation-reproduction-proof"

EXPECTED_OOF_SHA256 = "74a6332fcdeade31a33e15bcc225d59acf1f971cec5a61cd81b38c0505eaa8ba"
EXPECTED_ROWS = 160174
EXPECTED_LABEL_COUNTS = {0: 79971, 1: 80203}
EXPECTED_FOLD_COUNTS = {0: 53392, 1: 53391, 2: 53391}
EXPECTED_METRICS = {
    "average_precision": 0.8311228283421712,
    "brier_score": 0.169094052054208,
    "fraud_caught_at_3pct": 4743,
    "fraud_caught_at_5pct": 7845,
    "fraud_caught_at_7pct": 10857,
    "fraud_prevalence": 0.5007242124189943,
    "legitimate_audits_at_3pct": 62,
    "legitimate_audits_at_5pct": 163,
    "legitimate_audits_at_7pct": 355,
    "lift_at_3pct": 1.9713382131550867,
    "lift_at_5pct": 1.9564569284810422,
    "lift_at_7pct": 1.9338739200616288,
    "log_loss": 0.5069609155303831,
    "mean_prediction": 0.5015233781612578,
    "n_audited_at_3pct": 4805,
    "n_audited_at_5pct": 8008,
    "n_audited_at_7pct": 11212,
    "n_rows": 160174,
    "normalized_recall_at_3pct": 0.9870967741935484,
    "normalized_recall_at_5pct": 0.9796453546453546,
    "normalized_recall_at_7pct": 0.9683374955404923,
    "precision_at_3pct": 0.9870967741935484,
    "precision_at_5pct": 0.9796453546453546,
    "precision_at_7pct": 0.9683374955404923,
    "recall_at_3pct": 0.05913743874917397,
    "recall_at_5pct": 0.09781429622333329,
    "recall_at_7pct": 0.13536900115955763,
    "roc_auc": 0.8283982288959596,
}

if OOF_PATH is None:
    candidates = [
        Path.cwd() / OOF_FILENAME,
        Path.cwd() / "outputs" / "runs" / "foundation-v1" / "oof" / OOF_FILENAME,
        Path.cwd().parent / "outputs" / "runs" / "foundation-v1" / "oof" / OOF_FILENAME,
    ]
    OOF_PATH = next((path for path in candidates if path.exists()), None)
else:
    OOF_PATH = Path(OOF_PATH)

if OOF_PATH is None or not OOF_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {OOF_FILENAME}. Put it beside the notebook or set OOF_PATH."
    )

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print(f"Verifying: {OOF_PATH.resolve()}")
print(f"Evidence output: {OUTPUT_DIRECTORY.resolve()}")

## Recompute the metrics

The scoring code below is included in the notebook so the calculation is independent of the project. It uses stable descending ranking when determining the 3%, 5%, and 7% audit sets.

In [ ]:
def validate_arrays(labels, probabilities):
    labels = np.asarray(labels, dtype=int).reshape(-1)
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    if len(labels) != len(probabilities) or not len(labels):
        raise ValueError("Labels and probabilities must have the same non-zero length.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must contain only 0 and 1.")
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Probabilities must be finite values within [0, 1].")
    return labels, probabilities


def audit_metrics(labels, probabilities, audit_fraction):
    labels, probabilities = validate_arrays(labels, probabilities)
    audited_rows = int(np.floor(len(labels) * audit_fraction))
    selected = np.argsort(-probabilities, kind="mergesort")[:audited_rows]
    total_fraud = int(labels.sum())
    fraud_caught = int(labels[selected].sum())
    prevalence = total_fraud / len(labels)
    precision = fraud_caught / audited_rows if audited_rows else 0.0
    maximum_capturable = min(audited_rows, total_fraud)
    return {
        "n_audited": audited_rows,
        "fraud_caught": fraud_caught,
        "legitimate_audits": audited_rows - fraud_caught,
        "recall": fraud_caught / total_fraud if total_fraud else 0.0,
        "normalized_recall": fraud_caught / maximum_capturable if maximum_capturable else 0.0,
        "precision": precision,
        "lift": precision / prevalence if prevalence else 0.0,
    }


def evaluate_probabilities(labels, probabilities):
    labels, probabilities = validate_arrays(labels, probabilities)
    metrics = {
        "n_rows": len(labels),
        "fraud_prevalence": float(labels.mean()),
        "mean_prediction": float(probabilities.mean()),
        "brier_score": float(brier_score_loss(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
    }
    for fraction in (0.03, 0.05, 0.07):
        suffix = f"{fraction:.0%}".replace("%", "pct")
        for name, value in audit_metrics(labels, probabilities, fraction).items():
            metrics[f"{name}_at_{suffix}"] = value
    return metrics


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Verify the published result

PASS requires the exact OOF file fingerprint, expected row structure, and exact equality for every recorded metric.

In [ ]:
oof_sha256 = sha256_file(OOF_PATH)
oof = pd.read_csv(OOF_PATH)
required_columns = {"claim_id", "label", "fold", "fraud_probability_raw"}
missing_columns = sorted(required_columns - set(oof.columns))
if missing_columns:
    raise ValueError(f"OOF file is missing required columns: {missing_columns}")

labels = oof["label"].to_numpy(dtype=int)
probabilities = oof["fraud_probability_raw"].to_numpy(dtype=float)
actual_metrics = evaluate_probabilities(labels, probabilities)
structure_checks = {
    "sha256": oof_sha256 == EXPECTED_OOF_SHA256,
    "row_count": len(oof) == EXPECTED_ROWS,
    "unique_claim_ids": oof["claim_id"].is_unique,
    "label_counts": oof["label"].value_counts().sort_index().to_dict() == EXPECTED_LABEL_COUNTS,
    "fold_counts": oof["fold"].value_counts().sort_index().to_dict() == EXPECTED_FOLD_COUNTS,
}

comparison = pd.DataFrame(
    [
        {
            "metric": name,
            "expected": expected,
            "actual": actual_metrics.get(name),
            "exact_match": actual_metrics.get(name) == expected,
        }
        for name, expected in sorted(EXPECTED_METRICS.items())
    ]
)
metrics_match = bool(comparison["exact_match"].all())
structure_match = all(structure_checks.values())
passed = structure_match and metrics_match

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
verdict = {
    "status": "PASS" if passed else "FAIL",
    "candidate": CANDIDATE_NAME,
    "oof_path": str(OOF_PATH.resolve()),
    "oof_sha256": oof_sha256,
    "structure_checks": structure_checks,
    "metrics_exact_match": metrics_match,
    "environment": environment,
}

comparison.to_csv(OUTPUT_DIRECTORY / "reproduction_metric_comparison.csv", index=False)
with (OUTPUT_DIRECTORY / "reproduction_verdict.json").open("w") as handle:
    json.dump(verdict, handle, indent=2, sort_keys=True)

display(comparison)
display(pd.DataFrame([structure_checks]))
print(json.dumps({"status": verdict["status"], "output": str(OUTPUT_DIRECTORY.resolve())}, indent=2))

if not passed:
    raise AssertionError("Foundation result verification failed. See reproduction_verdict.json for details.")